# Add structure + loop type to the splits

Paste these cells after the split/tokenize notebook. They only rely on files that notebook wrote (`splits.csv`, and `train/val/test.csv` if you kept them), so they also work standalone.

1. Fold each unique sequence once with Arnie (ViennaRNA 2 MFE). Results are checkpointed, so an interrupted run resumes where it stopped.
2. Derive loop types from the dot-bracket.
3. Add `structure` and `loop_type` columns to `splits.csv` and to the full `train/val/test.csv` files.

Arnie needs its config file set up (the `ARNIEFILE` environment variable or `~/.arnie`) with the path to ViennaRNA.

In [1]:
from pathlib import Path
import os, random
from multiprocessing import Pool
import pandas as pd

OUTPUT_DIR = Path('preprocessed_rna')        # same folder the split notebook wrote to
N_WORKERS = 4
CHUNK = 5000                                # sequences folded between checkpoints
FOLD_PACKAGE = 'vienna_2'                      # ViennaRNA 2 MFE, as in the OpenVaccine pipeline
OPENVACCINE_JSON = None                      # e.g. Path('openvaccine/train.json') to validate the loop-type rules

FOLD_CHECKPOINT = OUTPUT_DIR / 'structures.csv'           # sequence, structure (raw fold results)
SEQ_FEATURES_CSV = OUTPUT_DIR / 'sequence_features.csv'   # sequence, structure, loop_type

## 1. Fold with Arnie

The smoke test runs first so a broken Arnie config fails in seconds instead of mid-run.

In [2]:
!pip install arnie
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -u -p /usr/local/miniconda

PREFIX=/usr/local/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local/miniconda


In [3]:
import os

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

In [6]:
! /usr/local/miniconda/bin/conda install -y -c conda-forge -c bioconda viennarna

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 26.9.4
    latest version: 26.10.0

Please update conda by running

    $ conda self update



## Package Plan ##

  environment location: /usr/local/miniconda

  added / updated specs:
    - viennarna


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.10.23 |       hbd8a1cb_0         127 KB  conda-forge
    certifi-2026.10.23         |     pyhd8ed1ab_3          13 KB  conda-forge
    conda-26.10.0              |  py314h9e666f3_1         1.5 MB  conda-forge
    conda-pycosat-solver-26.10

In [5]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [7]:
config = """vienna_2: /usr/local/miniconda/bin
TMP: /tmp
"""

with open("/content/arnie_config.txt", "w") as f:
    f.write(config)

In [8]:
import os

os.environ["ARNIEFILE"] = "/content/arnie_config.txt"
os.environ["vienna_2_PATH"] = "/usr/local/miniconda/bin"
os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

In [9]:
from arnie.utils import load_package_locations

print(load_package_locations())

{'vienna_2': '/usr/local/miniconda/bin', 'TMP': '/tmp'}


In [10]:
import os
from arnie.utils import load_package_locations

print("ARNIEFILE:", os.environ.get("ARNIEFILE"))
print("Config:")
!cat /content/arnie_config.txt

print("\nDetected packages:")
print(load_package_locations())

ARNIEFILE: /content/arnie_config.txt
Config:
vienna_2: /usr/local/miniconda/bin
TMP: /tmp

Detected packages:
{'vienna_2': '/usr/local/miniconda/bin', 'TMP': '/tmp'}


In [11]:
from arnie.mfe import mfe

def _init_worker():
    # forked workers start with identical random state; reseed in case Arnie derives temp-file names from it
    random.seed(os.urandom(8))

def fold_one(seq):
    try:
        return seq, mfe(seq, package=FOLD_PACKAGE)
    except Exception as e:
        # Log or print the sequence that failed to standard error, then return None
        import sys
        print(f"Skipping sequence due to RNAfold error: {seq[:30]}... ({e})", file=sys.stderr)
        return seq, None

test_seq = 'GGGAAACCCAUGGGAAAUCCCAUGCC'
test_structure = mfe(test_seq, package=FOLD_PACKAGE)
print(test_seq); print(test_structure)
assert isinstance(test_structure, str) and len(test_structure) == len(test_seq), 'Unexpected Arnie output'
assert set(test_structure) <= set('().'), 'Unexpected characters in structure'

GGGAAACCCAUGGGAAAUCCCAUGCC
((....))((((((....))))))..


In [12]:
from tqdm.notebook import tqdm

# Ensure the output directory exists so the checkpoint file can be written
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

splits = pd.read_csv('splits.csv', dtype={'sequence_id': str})
all_seqs = splits['sequence'].drop_duplicates().tolist()

done = set()
if FOLD_CHECKPOINT.exists():
    done = set(pd.read_csv(FOLD_CHECKPOINT, usecols=['sequence'])['sequence'])
todo = [s for s in all_seqs if s not in done]
print(f'{len(all_seqs):,} unique sequences; {len(done):,} already folded; {len(todo):,} to do')

write_header = not FOLD_CHECKPOINT.exists()
if todo:
    # Use tqdm to create a visual progress bar over the chunks
    with Pool(N_WORKERS, initializer=_init_worker) as pool:
        for start in tqdm(range(0, len(todo), CHUNK), desc="Folding Chunks"):
            chunk_seqs = todo[start:start + CHUNK]
            rows = pool.map(fold_one, chunk_seqs, chunksize=100)

            # Filter out any sequences where folding failed (returned None for structure)
            valid_rows = [row for row in rows if row[1] is not None]

            if valid_rows:
                pd.DataFrame(valid_rows, columns=['sequence', 'structure']).to_csv(
                    FOLD_CHECKPOINT, mode='a', header=write_header, index=False
                )
                write_header = False

167,684 unique sequences; 0 already folded; 167,684 to do


Folding Chunks:   0%|          | 0/34 [00:00<?, ?it/s]

Skipping sequence due to RNAfold error: GGGAACGACUCGAGUAGAGUCGAAAAGAGG... (RNAfold failed: on GGGAACGACUCGAGUAGAGUCGAAAAGAGGAGGCACCACGUGCUCUAUCCGUGCGGAAUCGACAUCAAGAGAUUUCGGAAGCAUAAUUUUUGGCGACCGGGGCAGCCGGUGAUCACGUUACCUGGCAUCGUAAGCGUAUUUCGAUACGCUUACGAUAAAAGAAACAACAACAACAAC
b'ERROR: Unexpected large magnitude discriminant = -1.70276 0.430111\n')
Skipping sequence due to RNAfold error: GGGAACGACUCGAGUAGAGUCGAAAAGAGG... (RNAfold failed: on GGGAACGACUCGAGUAGAGUCGAAAAGAGGAGGCACCACGUGCUCUAUCCGUGCGGAAUCGACAUCAAGAGAUUUCGGAAGCAUAAUUUUUGGCGACCGGGGCAGCCGGUGAUCACGUUACCUGGCGACGUGUUAUCUGUUCGCAGAUAACACGUCAAAAGAAACAACAACAACAAC
b'ERROR: Unexpected large magnitude discriminant = -1.70276 0.430111\n')
Skipping sequence due to RNAfold error: GGGAACGACUCGAGUAGAGUCGAAAAACCU... (RNAfold failed: on GGGAACGACUCGAGUAGAGUCGAAAAACCUUGAUGGUGUGAAGCGUACUACUAUAGAUCCUGAUCUUAGUAAAUACGUGAAGCGUGGUGACUACUACUUUACAGCUGCACCCCUAGAUGUGGUGGCGUUAUCUAAUUCGUUCGCGAAUUAGAUAACAAAAGAAACAACAACAACAAC
b'ERROR: Unexpected large magnitude di

## 2. Loop types from dot-bracket

Loop type is derived from the structure, not predicted. Labels follow bpRNA: `S` stem, `H` hairpin, `B` bulge, `I` internal loop, `M` multiloop, `X` external loop between stems, `E` 5'/3' end.

In [13]:
import os
# Check if the folder exists and list its contents
print("Folder exists:", os.path.exists('preprocessed_rna'))
if os.path.exists('preprocessed_rna'):
    print("Contents:", os.listdir('preprocessed_rna'))

Folder exists: True
Contents: ['structures.csv']


In [14]:
def loop_types(db: str) -> str:
    n = len(db)
    pair, stack = [-1] * n, []
    for i, c in enumerate(db):
        if c == '(':
            stack.append(i)
        elif c == ')':
            j = stack.pop()
            pair[i], pair[j] = j, i
        elif c != '.':
            raise ValueError(f'Unsupported character {c!r} (pseudoknots not handled)')
    if stack:
        raise ValueError('Unbalanced dot-bracket')

    out = [''] * n
    todo = [(0, n, False)]               # (start, end, closed by a pair?)
    while todo:
        start, end, closed = todo.pop()
        unpaired, children, i = [], [], start
        while i < end:
            if pair[i] == -1:
                unpaired.append(i); i += 1
            else:
                children.append((i, pair[i]))
                out[i] = out[pair[i]] = 'S'
                todo.append((i + 1, pair[i], True))
                i = pair[i] + 1
        if not closed:                   # exterior loop
            if not children:
                label = {u: 'E' for u in unpaired}
            else:
                first, last = children[0][0], children[-1][1]
                label = {u: 'E' if (u < first or u > last) else 'X' for u in unpaired}
        elif not children:
            label = {u: 'H' for u in unpaired}
        elif len(children) == 1:
            left = [u for u in unpaired if u < children[0][0]]
            right = [u for u in unpaired if u > children[0][1]]
            label = {u: 'I' if (left and right) else 'B' for u in unpaired}
        else:
            label = {u: 'M' for u in unpaired}
        for u, k in label.items():
            out[u] = k
    return ''.join(out)

# hand-checked cases
for db, want in {'((..))': 'SSHHSS', '.((.((...)).)).': 'ESSISSHHHSSISSE', '((.((...))))': 'SSBSSHHHSSSS',
                 '((..((...))..((...))..))': 'SSMMSSHHHSSMMSSHHHSSMMSS',
                 '((...))..((...))': 'SSHHHSSXXSSHHHSS', '.....': 'EEEEE'}.items():
    assert loop_types(db) == want, (db, loop_types(db), want)

# optional: compare against OpenVaccine's own structure -> predicted_loop_type columns
if OPENVACCINE_JSON:
    ov = pd.read_json(OPENVACCINE_JSON, lines=True)
    ok = sum(loop_types(s) == l for s, l in zip(ov['structure'], ov['predicted_loop_type']))
    print(f'OpenVaccine loop-type agreement: {ok}/{len(ov)} ({ok / len(ov):.2%})')

In [15]:
feats = pd.read_csv(FOLD_CHECKPOINT).drop_duplicates('sequence')
feats['loop_type'] = feats['structure'].map(loop_types)
feats.to_csv(SEQ_FEATURES_CSV, index=False)
print(feats.shape)
feats.head()

(167681, 3)


,sequence,structure,loop_type
0,GGGAACGACUCGAGUAGAGUCGAAAAAGAUCGCCACGCACUUACGA...,(.(((((((((.....)))))......((((((((((((((....)...,SISSSSSSSSSHHHHHSSSSSMMMMMMSSSSSSSSSSSSSSHHHHS...
1,GGGAACGACUCGAGUAGAGUCGAAAAGGUGGCCGGCAGAAUCGCGA...,.....((((((.....)))))).....(((.(((((.((...(......,EEEEESSSSSSHHHHHSSSSSSXXXXXSSSISSSSSISSIIISIII...
2,GGGAACGACUCGAGUAGAGUCGAAAACAUUGUUAAUGCCUAUAUUA...,.....((((((.....))))))........(((((.((((...(((...,EEEEESSSSSSHHHHHSSSSSSXXXXXXXXSSSSSISSSSIIISSS...
3,GGGAACGACUCGAGUAGAGUCGAAAAGGAGAUCGAAGACGACUUAC...,.....((((((.....))))))....((((((((..((((.(.......,EEEEESSSSSSHHHHHSSSSSSXXXXSSSSSSSSMMSSSSISHHHH...
4,GGGAACGACUCGAGUAGAGUCGAAAAGAUAUGGACUGACGAAGUCG...,.....((((((.....))))))....(((..(((((((((..((((...,EEEEESSSSSSHHHHHSSSSSSXXXXSSSIISSSSSSSSSBBSSSS...


## 3. Add the columns to every split

`splits.csv` gets `structure` and `loop_type` per `sequence_id`. The full `train/val/test.csv` files are rewritten in chunks with the same two columns inserted after `sequence`. Re-running is safe: existing columns are replaced, not duplicated.

In [22]:
assert feats['sequence'].is_unique
splits = pd.read_csv('splits.csv', dtype={'sequence_id': str}).drop(
    columns=['structure', 'predicted_loop_type', 'loop_type'], errors='ignore')

# Rename the 'loop_type' column in our features dataframe to 'predicted_loop_type'
feats_renamed = feats.rename(columns={'loop_type': 'predicted_loop_type'})

# Perform an inner merge to automatically drop sequences that were skipped during folding
splits = splits.merge(feats_renamed, on='sequence', how='inner', validate='many_to_one')
splits.to_csv('splits.csv', index=False)

structure_of = dict(zip(feats_renamed['sequence'], feats_renamed['structure']))
loop_of = dict(zip(feats_renamed['sequence'], feats_renamed['predicted_loop_type']))

for split in ['train', 'val', 'test']:
    # Wrap the string in Path so that .exists() and other path methods work correctly
    path = Path(f'{split}.csv')
    if not path.exists():
        print(f'{path.name} not found, skipping'); continue
    tmp, first, n_rows = path.with_suffix('.tmp'), True, 0
    for chunk in pd.read_csv(path, chunksize=10_000, low_memory=False, dtype={'sequence_id': str}):
        chunk = chunk.drop(columns=['structure', 'predicted_loop_type', 'loop_type'], errors='ignore')

        # Filter chunk rows to only keep sequences that successfully folded
        chunk = chunk[chunk['sequence'].isin(structure_of)]

        pos = chunk.columns.get_loc('sequence') + 1
        new_cols = pd.DataFrame({'structure': chunk['sequence'].map(structure_of),
                                 'predicted_loop_type': chunk['sequence'].map(loop_of)}, index=chunk.index)
        chunk = pd.concat([chunk.iloc[:, :pos], new_cols, chunk.iloc[:, pos:]], axis=1)
        assert chunk[['structure', 'predicted_loop_type']].notna().all().all()
        chunk.to_csv(tmp, mode='w' if first else 'a', header=first, index=False)
        first, n_rows = False, n_rows + len(chunk)
    os.replace(tmp, path)
    print(f'{path.name}: {n_rows:,} rows updated')

train.csv: 301,826 rows updated
val.csv: 16,768 rows updated
test.csv: 16,768 rows updated


## 4. Sanity checks

In [26]:
from collections import Counter

splits = pd.read_csv('splits.csv', dtype={'sequence_id': str})
for split, part in splits.groupby('split'):
    assert (part['sequence'].str.len() == part['structure'].str.len()).all()
    assert (part['structure'].str.len() == part['predicted_loop_type'].str.len()).all()
    print(split, len(part), 'rows OK')

# spot-check consistency on a sample: S exactly where the structure has a pair
sample = feats.sample(min(20_000, len(feats)), random_state=0)
assert set(''.join(sample['structure'])) <= set('().')
for db, lt in zip(sample['structure'], sample['loop_type']):
    assert all((l == 'S') == (d != '.') for d, l in zip(db, lt))

counts = Counter(''.join(sample['loop_type']))
total = sum(counts.values())
print({k: f'{v / total:.1%}' for k, v in counts.most_common()})
splits.head()

test 8384 rows OK
train 150913 rows OK
val 8384 rows OK
{'S': '56.7%', 'H': '12.6%', 'E': '12.5%', 'I': '6.5%', 'X': '5.5%', 'M': '4.4%', 'B': '1.7%'}


,sequence_id,sequence,split,structure,predicted_loop_type
0,0000d87cab97,GGGAACGACUCGAGUAGAGUCGAAAAAGAUCGCCACGCACUUACGA...,train,(.(((((((((.....)))))......((((((((((((((....)...,SISSSSSSSSSHHHHHSSSSSMMMMMMSSSSSSSSSSSSSSHHHHS...
1,0001ca9d21b0,GGGAACGACUCGAGUAGAGUCGAAAAGGUGGCCGGCAGAAUCGCGA...,train,.....((((((.....)))))).....(((.(((((.((...(......,EEEEESSSSSSHHHHHSSSSSSXXXXXSSSISSSSSISSIIISIII...
2,00021f968267,GGGAACGACUCGAGUAGAGUCGAAAACAUUGUUAAUGCCUAUAUUA...,train,.....((((((.....))))))........(((((.((((...(((...,EEEEESSSSSSHHHHHSSSSSSXXXXXXXXSSSSSISSSSIIISSS...
3,00026ef17e1b,GGGAACGACUCGAGUAGAGUCGAAAAGGAGAUCGAAGACGACUUAC...,train,.....((((((.....))))))....((((((((..((((.(.......,EEEEESSSSSSHHHHHSSSSSSXXXXSSSSSSSSMMSSSSISHHHH...
4,000302908510,GGGAACGACUCGAGUAGAGUCGAAAAGAUAUGGACUGACGAAGUCG...,train,.....((((((.....))))))....(((..(((((((((..((((...,EEEEESSSSSSHHHHHSSSSSSXXXXSSSIISSSSSSSSSBBSSSS...


In [3]:
import pandas as pd
from pathlib import Path
import os

# Ensure we read the sequence features that only contain successfully folded results (167,681 rows)
feats = pd.read_csv('sequence_features.csv').drop_duplicates('sequence')
assert len(feats) == 167681, f"Expected 167,681 features, found {len(feats)}"

# 1. Clean and rewrite splits.csv to keep only successfully folded sequences
splits = pd.read_csv('splits.csv', dtype={'sequence_id': str}).drop(
    columns=['structure', 'predicted_loop_type', 'loop_type'], errors='ignore'
)

# Inner merge to drop the 3 failed sequences
feats_renamed = feats.rename(columns={'loop_type': 'predicted_loop_type'})
splits = splits.merge(feats_renamed, on='sequence', how='inner', validate='many_to_one')
splits.to_csv('splits.csv', index=False)
print(f"Overwrote splits.csv with {len(splits):,} rows.")

structure_of = dict(zip(feats_renamed['sequence'], feats_renamed['structure']))
loop_of = dict(zip(feats_renamed['sequence'], feats_renamed['predicted_loop_type']))

# 2. Re-write the train/val/test splits to drop those 3 sequences
for split in ['train', 'val', 'test']:
    path = Path(f'{split}.csv')
    if not path.exists():
        print(f'{path.name} not found, skipping'); continue
    tmp, first, n_rows = path.with_suffix('.tmp'), True, 0
    for chunk in pd.read_csv(path, chunksize=10_000, low_memory=False, dtype={'sequence_id': str}):
        chunk = chunk.drop(columns=['structure', 'predicted_loop_type', 'loop_type'], errors='ignore')

        # Filter out the 3 failed sequences
        chunk = chunk[chunk['sequence'].isin(structure_of)]

        pos = chunk.columns.get_loc('sequence') + 1
        new_cols = pd.DataFrame({'structure': chunk['sequence'].map(structure_of),
                                 'predicted_loop_type': chunk['sequence'].map(loop_of)}, index=chunk.index)
        chunk = pd.concat([chunk.iloc[:, :pos], new_cols, chunk.iloc[:, pos:]], axis=1)
        assert chunk[['structure', 'predicted_loop_type']].notna().all().all()
        chunk.to_csv(tmp, mode='w' if first else 'a', header=first, index=False)
        first, n_rows = False, n_rows + len(chunk)
    os.replace(tmp, path)
    print(f'{path.name}: {n_rows:,} rows updated')

Overwrote splits.csv with 167,681 rows.
train.csv: 301,826 rows updated
val.csv: 16,768 rows updated
test.csv: 16,768 rows updated


In [5]:
# 3. Run validation assertions
print("=== Running Deletion & Row Count Verification ===")
splits_df = pd.read_csv('splits.csv')
unique_in_splits = splits_df['sequence'].nunique()
print(f"Unique sequences in splits.csv: {unique_in_splits:,} (Expected: 167,681)")
assert unique_in_splits == 167681, f"Mismatch! Found {unique_in_splits} unique sequences."

splits_seq_ids = set(splits_df['sequence_id'].dropna().unique())
individual_splits_seq_ids = set()

for split in ['train', 'val', 'test']:
    path = Path(f'{split}.csv')
    if path.exists():
        df_split = pd.read_csv(path, dtype={'sequence_id': str})
        unique_split_seqs = df_split['sequence'].nunique()
        individual_splits_seq_ids.update(df_split['sequence_id'].dropna().unique())
        print(f" - {split}.csv: {len(df_split):,} rows, {unique_split_seqs:,} unique sequences.")
        assert df_split['structure'].notna().all(), f"NaN values found in structure for {split}.csv"
        assert df_split['predicted_loop_type'].notna().all(), f"NaN values found in predicted_loop_type for {split}.csv"

# Confirm that the exactly matched sequence_ids align between our files
assert splits_seq_ids == individual_splits_seq_ids, "Sequence ID mismatch between splits.csv and sub-split files!"
print("✔ Verified: The sequence IDs are perfectly identical and aligned between splits.csv and sub-split files.")
print("✔ Verified: Failed sequences are completely dropped across all files.")

=== Running Deletion & Row Count Verification ===
Unique sequences in splits.csv: 167,681 (Expected: 167,681)
 - train.csv: 301,826 rows, 150,913 unique sequences.
 - val.csv: 16,768 rows, 8,384 unique sequences.
 - test.csv: 16,768 rows, 8,384 unique sequences.
✔ Verified: The sequence IDs are perfectly identical and aligned between splits.csv and sub-split files.
✔ Verified: Failed sequences are completely dropped across all files.


In [6]:
# 4. Confirm loop type definitions according to bpRNA taxonomy
print("=== bpRNA Terminology & Rule Validation ===")
sample_data = splits_df.sample(min(10000, len(splits_df)), random_state=42)

valid_chars = set("SIBHMXE")
for idx, row in sample_data.iterrows():
    seq, struct, loop = row['sequence'], row['structure'], row['predicted_loop_type']

    # Length consistency check
    assert len(seq) == len(struct) == len(loop), f"Length mismatch at index {idx}"

    # Character set check
    invalid_loop_chars = set(loop) - valid_chars
    assert not invalid_loop_chars, f"Found unexpected loop characters: {invalid_loop_chars}"

    # Stem rule: 'S' must exist if and only if there is a base pair '(' or ')'
    for s_char, l_char in zip(struct, loop):
        if s_char in '()':
            assert l_char == 'S', "Structure has a pair but loop type is not 'S'"
        else:
            assert l_char != 'S', "Structure is unpaired but loop type is 'S'"

print("✔ Verified: Loop types are successfully generated and conform perfectly to bpRNA structural rules.")

=== bpRNA Terminology & Rule Validation ===
✔ Verified: Loop types are successfully generated and conform perfectly to bpRNA structural rules.
